# Code 2.8.1: A complete training loop with early stopping

Data splitting, evaluation, and a `fit` function that trains with minibatch SGD, evaluates on the validation set after every epoch, keeps the best checkpoint, and stops early, followed by a single evaluation on the test set. It relies on `init_params`, `forward`, `softmax_cross_entropy`, and `backward` from Section 2.7 (Code 2.7.2, also in `figures/src/numpy_mlp.py`) and on `make_moons` from `figures/src/style.py`.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.
# Needed by make_moons
import numpy as np

# From Code 2.7.2
def init_params(n_in, n_hidden, n_out, seed=0):
    rng = np.random.default_rng(seed)
    return {
        "W1": rng.normal(0, np.sqrt(2 / n_in), (n_in, n_hidden)),
        "b1": np.zeros(n_hidden),
        "W2": rng.normal(0, np.sqrt(1 / n_hidden), (n_hidden, n_out)),
        "b2": np.zeros(n_out),
    }

def forward(P, X):
    Z1 = X @ P["W1"] + P["b1"]          # (B, H)   bias broadcasts over rows
    H = np.maximum(0, Z1)               # (B, H)
    Z2 = H @ P["W2"] + P["b2"]          # (B, C)   logits
    return Z2, (X, Z1, H)

def softmax_cross_entropy(Z, y):
    """Mean cross-entropy of integer labels y under softmax(Z); also returns dL/dZ."""
    Zs = Z - Z.max(axis=1, keepdims=True)        # stability: subtract the row max
    logsumexp = np.log(np.exp(Zs).sum(axis=1, keepdims=True))
    logp = Zs - logsumexp                        # log-softmax, (B, C)
    B = len(y)
    loss = -logp[np.arange(B), y].mean()
    dZ = np.exp(logp)                            # probabilities
    dZ[np.arange(B), y] -= 1.0                   # p - onehot(y)
    return loss, dZ / B

def backward(P, cache, dZ2):
    X, Z1, H = cache
    grads = {}
    grads["W2"] = H.T @ dZ2                      # (H, B) @ (B, C) -> (H, C)
    grads["b2"] = dZ2.sum(axis=0)                # (C,)
    dH = dZ2 @ P["W2"].T                         # (B, C) @ (C, H) -> (B, H)
    dZ1 = dH * (Z1 > 0)                          # ReLU passes gradient where Z1 > 0
    grads["W1"] = X.T @ dZ1                      # (D, B) @ (B, H) -> (D, H)
    grads["b1"] = dZ1.sum(axis=0)                # (H,)
    for k in P:
        assert grads[k].shape == P[k].shape, k   # every gradient matches its parameter
    return grads

# From chapters/02-neural-network-basics/figures/src/style.py
def make_moons(n=200, noise=0.15, seed=0):
    """Two interleaving half-circles (our own implementation, no sklearn needed)."""
    rng = np.random.default_rng(seed)
    n0 = n // 2
    n1 = n - n0
    t0 = rng.uniform(0, np.pi, n0)
    t1 = rng.uniform(0, np.pi, n1)
    X0 = np.c_[np.cos(t0), np.sin(t0)]
    X1 = np.c_[1 - np.cos(t1), 0.5 - np.sin(t1)]
    X = np.r_[X0, X1] + rng.normal(0, noise, (n, 2))
    y = np.r_[np.zeros(n0, int), np.ones(n1, int)]
    perm = rng.permutation(n)
    return X[perm], y[perm]


In [ ]:
import copy
import numpy as np

def split(X, y, frac_val=0.15, frac_test=0.15, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(X))
    n_test, n_val = int(frac_test * len(X)), int(frac_val * len(X))
    test, val, train = idx[:n_test], idx[n_test:n_test + n_val], idx[n_test + n_val:]
    return (X[train], y[train]), (X[val], y[val]), (X[test], y[test])

def evaluate(P, X, y):
    Z, _ = forward(P, X)
    loss, _ = softmax_cross_entropy(Z, y)
    return loss, np.mean(Z.argmax(axis=1) == y)

def fit(train, val, n_hidden=128, lr=0.1, batch_size=16, max_epochs=500, patience=50, seed=0):
    (X_tr, y_tr), (X_val, y_val) = train, val
    rng = np.random.default_rng(seed)
    P = init_params(X_tr.shape[1], n_hidden, int(y_tr.max()) + 1, seed=seed)
    best = {"val_loss": np.inf, "epoch": -1, "params": None}
    history = []
    for epoch in range(max_epochs):
        order = rng.permutation(len(X_tr))                 # reshuffle every epoch
        for start in range(0, len(X_tr), batch_size):
            idx = order[start:start + batch_size]          # 1. sample a minibatch
            Z, cache = forward(P, X_tr[idx])               # 2. forward pass
            loss, dZ = softmax_cross_entropy(Z, y_tr[idx]) # 3. loss (and dL/dZ)
            grads = backward(P, cache, dZ)                 # 4. backward pass
            for k in P:                                    # 5. update
                P[k] -= lr * grads[k]
        tr_loss, _ = evaluate(P, X_tr, y_tr)
        val_loss, val_acc = evaluate(P, X_val, y_val)
        history.append((epoch, tr_loss, val_loss))
        if val_loss < best["val_loss"]:                    # keep the best checkpoint
            best = {"val_loss": val_loss, "epoch": epoch, "params": copy.deepcopy(P)}
        elif epoch - best["epoch"] >= patience:            # early stopping
            break
    return best, history

X, y = make_moons(n=300, noise=0.35, seed=5)
train, val, test = split(X, y)
best, history = fit(train, val)
last_epoch, last_train, last_val = history[-1]
print(f"stopped after epoch {last_epoch}: train loss {last_train:.3f}, val loss {last_val:.3f}")
print(f"best epoch {best['epoch']}: val loss {best['val_loss']:.3f}")
test_loss, test_acc = evaluate(best["params"], *test)    # touched once, at the very end
print(f"test loss {test_loss:.3f}, test accuracy {test_acc:.2f}")


**Expected output**

Output:

```text
stopped after epoch 281: train loss 0.240, val loss 0.343
best epoch 231: val loss 0.226
test loss 0.229, test accuracy 0.91
```
